# Notebook 4 - Patrol Optimizer

This notebook converts hotspot analytics into an enforcement action plan.

It combines:
- `priority_score`
- `risk_level`
- `trajectory`
- `anomaly_status`

into a single `patrol_priority` score, then generates:
- Top 20 Enforcement Zones
- Best Patrol Windows

This is the operational answer to the hackathon prompt: how AI-driven parking intelligence can enable targeted enforcement.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

## Load Inputs

In [2]:
hotspot_events = pd.read_csv("hotspot_clustered.csv")
cluster_stats = pd.read_csv("hotspot_cluster_stats.csv")
trajectory_df = pd.read_csv("trajectory_analysis.csv")
anomaly_df = pd.read_csv("anomaly_analysis.csv")

print(hotspot_events.shape)
print(cluster_stats.shape)
print(trajectory_df.shape)
print(anomaly_df.shape)

(240654, 34)
(506, 20)
(506, 4)
(506, 6)


## Build Patrol Priority

In [3]:
patrol_df = (
    cluster_stats
    .merge(
        trajectory_df,
        on="cluster_id",
        how="left"
    )
    .merge(
        anomaly_df,
        on="cluster_id",
        how="left"
    )
)

trajectory_base_map = {
    "Declining": 0.25,
    "Stable": 0.50,
    "Escalating": 1.00,
    "Insufficient History": 0.10,
}

anomaly_base_map = {
    "Normal": 0.40,
    "Abnormal Surge": 1.00,
    "Abnormal Drop": 0.20,
}

patrol_df["trajectory_base_score"] = (
    patrol_df["trajectory"]
    .map(trajectory_base_map)
    .fillna(0.25)
)

patrol_df["priority_norm"] = MinMaxScaler().fit_transform(
    patrol_df[["priority_score"]]
)

patrol_df["r2_filled"] = patrol_df["r2"].fillna(0)
patrol_df["trajectory_weight"] = (
    patrol_df["trajectory_base_score"]
    * (0.50 + 0.50 * patrol_df["r2_filled"])
)

patrol_df["anomaly_base_score"] = (
    patrol_df["anomaly_status"]
    .map(anomaly_base_map)
    .fillna(0.40)
)

patrol_df["surge_intensity"] = patrol_df["z_score"].clip(lower=0).fillna(0)
patrol_df["surge_intensity_norm"] = MinMaxScaler().fit_transform(
    patrol_df[["surge_intensity"]]
)

patrol_df["anomaly_weight"] = (
    0.60 * patrol_df["anomaly_base_score"]
    + 0.40 * patrol_df["surge_intensity_norm"]
)

patrol_df["patrol_priority"] = (
    0.50 * patrol_df["priority_norm"]
    + 0.30 * patrol_df["trajectory_weight"]
    + 0.20 * patrol_df["anomaly_weight"]
)

patrol_df[[
    "cluster_id",
    "priority_score",
    "priority_norm",
    "risk_level",
    "trajectory",
    "anomaly_status",
    "patrol_priority"
]].head()

,cluster_id,priority_score,priority_norm,risk_level,trajectory,anomaly_status,patrol_priority
0,0.0,0.377956,0.514532,Medium,Stable,Normal,0.380267
1,1.0,0.488038,0.702897,High,Escalating,Abnormal Surge,0.686695
2,2.0,0.614845,0.919881,Critical,Declining,Normal,0.584285
3,3.0,0.584005,0.867110,Critical,Declining,Normal,0.572742
4,4.0,0.490399,0.706937,High,Declining,Normal,0.466451


## Top 20 Enforcement Zones

In [4]:
top_enforcement_zones = (
    patrol_df[
        [
            "cluster_id",
            "top_station",
            "centroid_lat",
            "centroid_lon",
            "violations",
            "avg_severity",
            "active_months",
            "priority_score",
            "risk_level",
            "trajectory",
            "r2",
            "current_month",
            "z_score",
            "anomaly_status",
            "patrol_priority"
        ]
    ]
    .sort_values(
        "patrol_priority",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_enforcement_zones

,cluster_id,top_station,centroid_lat,centroid_lon,violations,avg_severity,active_months,priority_score,risk_level,trajectory,r2,current_month,z_score,anomaly_status,patrol_priority
0,114.0,Mahadevapura,12.996312,77.668619,1537,3.797658,6,0.660232,Critical,Escalating,0.000336,486.0,1.676563,Abnormal Surge,0.834530
1,6.0,K.R. Pura,13.008441,77.695320,3356,2.640644,6,0.617709,Critical,Escalating,0.179653,1030.0,1.712017,Abnormal Surge,0.826436
2,187.0,Whitefield,12.951403,77.717501,349,4.272206,6,0.594333,Critical,Escalating,0.262344,177.0,1.863662,Abnormal Surge,0.824784
3,31.0,Malleshwaram,13.010297,77.553283,8563,1.635642,6,0.569432,Critical,Escalating,0.057540,2127.0,1.664340,Abnormal Surge,0.764946
4,248.0,Hulimavu,12.885738,77.641532,465,1.997849,6,0.509601,High,Escalating,0.155951,204.0,1.984256,Abnormal Surge,0.741056
5,39.0,HAL Old Airport,12.948252,77.699061,1542,3.484436,6,0.643953,Critical,Escalating,0.034360,399.0,0.936182,Normal,0.724689
6,54.0,Kodigehalli,13.047512,77.575880,866,1.764434,6,0.483510,High,Escalating,0.775036,266.0,1.352130,Normal,0.714822
7,422.0,K.S. Layout,12.911473,77.552080,35,4.457143,5,0.483789,High,Escalating,0.092048,21.0,1.831613,Abnormal Surge,0.703404
8,50.0,Banaswadi,12.993636,77.662697,760,1.436842,6,0.454326,High,Escalating,0.193301,320.0,1.786767,Abnormal Surge,0.691628
9,23.0,Mahadevapura,13.000354,77.676957,1623,2.723352,6,0.594344,Critical,Escalating,0.034651,462.0,1.132262,Normal,0.689974


## Best Patrol Windows

## Dashboard Centerpiece

In [5]:
top_enforcement_zones[
    [
        "cluster_id",
        "top_station",
        "risk_level",
        "trajectory",
        "anomaly_status",
        "patrol_priority"
    ]
].head(20)

,cluster_id,top_station,risk_level,trajectory,anomaly_status,patrol_priority
0,114.0,Mahadevapura,Critical,Escalating,Abnormal Surge,0.834530
1,6.0,K.R. Pura,Critical,Escalating,Abnormal Surge,0.826436
2,187.0,Whitefield,Critical,Escalating,Abnormal Surge,0.824784
3,31.0,Malleshwaram,Critical,Escalating,Abnormal Surge,0.764946
4,248.0,Hulimavu,High,Escalating,Abnormal Surge,0.741056
5,39.0,HAL Old Airport,Critical,Escalating,Normal,0.724689
6,54.0,Kodigehalli,High,Escalating,Normal,0.714822
7,422.0,K.S. Layout,High,Escalating,Abnormal Surge,0.703404
8,50.0,Banaswadi,High,Escalating,Abnormal Surge,0.691628
9,23.0,Mahadevapura,Critical,Escalating,Normal,0.689974


In [6]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

top_cluster_ids = top_enforcement_zones["cluster_id"]

window_df = hotspot_events[
    hotspot_events["cluster_id"].isin(top_cluster_ids)
].copy()

window_df["day_of_week"] = pd.Categorical(
    window_df["day_of_week"],
    categories=day_order,
    ordered=True
)

window_df = window_df.dropna(subset=["hour", "day_of_week"]).copy()
window_df["hour"] = window_df["hour"].astype(int)

cluster_totals = (
    window_df
    .groupby("cluster_id")
    .agg(cluster_total_violations=("id", "count"))
    .reset_index()
)

window_profiles = (
    window_df
    .groupby(
        ["cluster_id", "day_of_week", "hour"],
        observed=True
    )
    .agg(
        window_violations=("id", "count"),
        avg_window_severity=("max_severity", "mean"),
        weekend_share=("is_weekend", "mean")
    )
    .reset_index()
    .merge(
        cluster_totals,
        on="cluster_id",
        how="left"
    )
)

window_profiles["hour_percentage"] = (
    window_profiles["window_violations"]
    / window_profiles["cluster_total_violations"]
)

window_profiles["window_violations_norm"] = MinMaxScaler().fit_transform(
    window_profiles[["window_violations"]]
)
window_profiles["avg_window_severity_norm"] = MinMaxScaler().fit_transform(
    window_profiles[["avg_window_severity"]]
)

window_profiles["window_score"] = (
    0.75 * window_profiles["window_violations_norm"]
    + 0.25 * window_profiles["avg_window_severity_norm"]
)

window_profiles["window_rank"] = (
    window_profiles
    .groupby("cluster_id")["window_score"]
    .rank(method="first", ascending=False)
)

window_profiles["patrol_window"] = (
    window_profiles["day_of_week"].astype(str)
    + " "
    + window_profiles["hour"].map(lambda x: f"{x:02d}:00-{(x + 1):02d}:00")
)

best_windows_detailed = (
    window_profiles
    .sort_values(
        ["cluster_id", "window_score"],
        ascending=[True, False]
    )
    .query("window_rank <= 3")
    .copy()
)
best_windows_detailed["hour_percentage_label"] = (
    (100 * best_windows_detailed["hour_percentage"])
    .round(1)
    .astype(str)
    + "% of hotspot activity"
)

best_windows_detailed.head(20)

,cluster_id,day_of_week,hour,window_violations,avg_window_severity,weekend_share,cluster_total_violations,hour_percentage,window_violations_norm,avg_window_severity_norm,window_score,window_rank,patrol_window,hour_percentage_label
31,1.0,Thursday,9,1,6.000000,0.0,427,0.002342,0.000000,0.833333,0.208333,1.0,Thursday 09:00-10:00,0.2% of hotspot activity
4,1.0,Monday,18,2,5.500000,0.0,427,0.004684,0.002959,0.750000,0.189719,2.0,Monday 18:00-19:00,0.5% of hotspot activity
43,1.0,Friday,9,9,5.000000,0.0,427,0.021077,0.023669,0.666667,0.184418,3.0,Friday 09:00-10:00,2.1% of hotspot activity
202,6.0,Sunday,18,152,3.085526,1.0,3356,0.045292,0.446746,0.347588,0.421956,1.0,Sunday 18:00-19:00,4.5% of hotspot activity
203,6.0,Sunday,19,144,2.645833,1.0,3356,0.042908,0.423077,0.274306,0.385884,2.0,Sunday 19:00-20:00,4.3% of hotspot activity
164,6.0,Friday,19,120,2.983333,0.0,3356,0.035757,0.352071,0.330556,0.346692,3.0,Friday 19:00-20:00,3.6% of hotspot activity
263,20.0,Sunday,5,143,2.678322,1.0,1095,0.130594,0.420118,0.279720,0.385019,1.0,Sunday 05:00-06:00,13.1% of hotspot activity
261,20.0,Sunday,3,120,3.833333,1.0,1095,0.109589,0.352071,0.472222,0.382109,2.0,Sunday 03:00-04:00,11.0% of hotspot activity
264,20.0,Sunday,6,148,2.250000,1.0,1095,0.135160,0.434911,0.208333,0.378267,3.0,Sunday 06:00-07:00,13.5% of hotspot activity
305,23.0,Thursday,2,57,5.298246,0.0,1623,0.035120,0.165680,0.716374,0.303354,1.0,Thursday 02:00-03:00,3.5% of hotspot activity


In [7]:
best_windows_summary = (
    best_windows_detailed
    .assign(
        patrol_window_summary=(
            best_windows_detailed["patrol_window"]
            + " ("
            + best_windows_detailed["hour_percentage_label"]
            + ")"
        )
    )
    .groupby("cluster_id")
    .agg(
        recommended_windows=(
            "patrol_window_summary",
            lambda x: " | ".join(x)
        )
    )
    .reset_index()
)

best_patrol_windows = (
    top_enforcement_zones
    .merge(
        best_windows_summary,
        on="cluster_id",
        how="left"
    )
    [[
        "cluster_id",
        "top_station",
        "risk_level",
        "trajectory",
        "anomaly_status",
        "patrol_priority",
        "recommended_windows"
    ]]
)

best_patrol_windows

,cluster_id,top_station,risk_level,trajectory,anomaly_status,patrol_priority,recommended_windows
0,114.0,Mahadevapura,Critical,Escalating,Abnormal Surge,0.834530,Friday 04:00-05:00 (4.6% of hotspot activity) ...
1,6.0,K.R. Pura,Critical,Escalating,Abnormal Surge,0.826436,Sunday 18:00-19:00 (4.5% of hotspot activity) ...
2,187.0,Whitefield,Critical,Escalating,Abnormal Surge,0.824784,Thursday 03:00-04:00 (14.0% of hotspot activit...
3,31.0,Malleshwaram,Critical,Escalating,Abnormal Surge,0.764946,Sunday 04:00-05:00 (4.0% of hotspot activity) ...
4,248.0,Hulimavu,High,Escalating,Abnormal Surge,0.741056,Sunday 20:00-21:00 (10.5% of hotspot activity)...
5,39.0,HAL Old Airport,Critical,Escalating,Normal,0.724689,Thursday 22:00-23:00 (6.0% of hotspot activity...
6,54.0,Kodigehalli,High,Escalating,Normal,0.714822,Wednesday 21:00-22:00 (6.5% of hotspot activit...
7,422.0,K.S. Layout,High,Escalating,Abnormal Surge,0.703404,Monday 20:00-21:00 (11.4% of hotspot activity)...
8,50.0,Banaswadi,High,Escalating,Abnormal Surge,0.691628,Monday 09:00-10:00 (5.9% of hotspot activity) ...
9,23.0,Mahadevapura,Critical,Escalating,Normal,0.689974,Thursday 02:00-03:00 (3.5% of hotspot activity...


## Final Enforcement Action Plan

In [8]:
patrol_action_plan = (
    top_enforcement_zones
    .merge(
        best_windows_summary,
        on="cluster_id",
        how="left"
    )
)

patrol_action_plan["deployment_level"] = pd.cut(
    patrol_action_plan["patrol_priority"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=[
        "Routine Monitoring",
        "Targeted Patrol",
        "Immediate Enforcement"
    ],
    include_lowest=True
)

patrol_action_plan[[
    "cluster_id",
    "risk_level",
    "trajectory",
    "anomaly_status",
    "patrol_priority",
    "deployment_level",
    "recommended_windows"
]]

,cluster_id,risk_level,trajectory,anomaly_status,patrol_priority,deployment_level,recommended_windows
0,114.0,Critical,Escalating,Abnormal Surge,0.834530,Immediate Enforcement,Friday 04:00-05:00 (4.6% of hotspot activity) ...
1,6.0,Critical,Escalating,Abnormal Surge,0.826436,Immediate Enforcement,Sunday 18:00-19:00 (4.5% of hotspot activity) ...
2,187.0,Critical,Escalating,Abnormal Surge,0.824784,Immediate Enforcement,Thursday 03:00-04:00 (14.0% of hotspot activit...
3,31.0,Critical,Escalating,Abnormal Surge,0.764946,Immediate Enforcement,Sunday 04:00-05:00 (4.0% of hotspot activity) ...
4,248.0,High,Escalating,Abnormal Surge,0.741056,Immediate Enforcement,Sunday 20:00-21:00 (10.5% of hotspot activity)...
5,39.0,Critical,Escalating,Normal,0.724689,Immediate Enforcement,Thursday 22:00-23:00 (6.0% of hotspot activity...
6,54.0,High,Escalating,Normal,0.714822,Immediate Enforcement,Wednesday 21:00-22:00 (6.5% of hotspot activit...
7,422.0,High,Escalating,Abnormal Surge,0.703404,Immediate Enforcement,Monday 20:00-21:00 (11.4% of hotspot activity)...
8,50.0,High,Escalating,Abnormal Surge,0.691628,Targeted Patrol,Monday 09:00-10:00 (5.9% of hotspot activity) ...
9,23.0,Critical,Escalating,Normal,0.689974,Targeted Patrol,Thursday 02:00-03:00 (3.5% of hotspot activity...


## Save Outputs

In [9]:
top_enforcement_zones.to_csv(
    "top_enforcement_zones.csv",
    index=False
)

best_windows_detailed.to_csv(
    "best_patrol_windows_detailed.csv",
    index=False
)

best_patrol_windows.to_csv(
    "best_patrol_windows.csv",
    index=False
)

patrol_action_plan.to_csv(
    "patrol_action_plan.csv",
    index=False
)

print("Saved")

Saved


In [10]:
patrol_action_plan["deployment_level"].value_counts()

deployment_level
Targeted Patrol          12
Immediate Enforcement     8
Routine Monitoring        0
Name: count, dtype: int64

In [11]:
patrol_action_plan[
[
    "cluster_id",
    "risk_level",
    "trajectory",
    "anomaly_status",
    "patrol_priority",
    "deployment_level",
    "recommended_windows"
]
].head(20)

,cluster_id,risk_level,trajectory,anomaly_status,patrol_priority,deployment_level,recommended_windows
0,114.0,Critical,Escalating,Abnormal Surge,0.834530,Immediate Enforcement,Friday 04:00-05:00 (4.6% of hotspot activity) ...
1,6.0,Critical,Escalating,Abnormal Surge,0.826436,Immediate Enforcement,Sunday 18:00-19:00 (4.5% of hotspot activity) ...
2,187.0,Critical,Escalating,Abnormal Surge,0.824784,Immediate Enforcement,Thursday 03:00-04:00 (14.0% of hotspot activit...
3,31.0,Critical,Escalating,Abnormal Surge,0.764946,Immediate Enforcement,Sunday 04:00-05:00 (4.0% of hotspot activity) ...
4,248.0,High,Escalating,Abnormal Surge,0.741056,Immediate Enforcement,Sunday 20:00-21:00 (10.5% of hotspot activity)...
5,39.0,Critical,Escalating,Normal,0.724689,Immediate Enforcement,Thursday 22:00-23:00 (6.0% of hotspot activity...
6,54.0,High,Escalating,Normal,0.714822,Immediate Enforcement,Wednesday 21:00-22:00 (6.5% of hotspot activit...
7,422.0,High,Escalating,Abnormal Surge,0.703404,Immediate Enforcement,Monday 20:00-21:00 (11.4% of hotspot activity)...
8,50.0,High,Escalating,Abnormal Surge,0.691628,Targeted Patrol,Monday 09:00-10:00 (5.9% of hotspot activity) ...
9,23.0,Critical,Escalating,Normal,0.689974,Targeted Patrol,Thursday 02:00-03:00 (3.5% of hotspot activity...
